In [ ]:
import copy


# Estimating Bias Between Sensors


This example demonstrates how to simulate and estimate a drifting bias in the position of a sensor platform.
Specifically, the platform at index 0 (and its sensor) will have a time-varying bias applied to its position.
We use Stone-Soup's bias wrappers, feeders and updater to estimate this changing bias from sensor measurements.



In [ ]:
# Some initial imports and set up
import datetime
import numpy as np

np.random.seed(2001)
start_time = datetime.datetime.now().replace(microsecond=0)

## Define Platforms and Sensors

We create three moving platforms, each with a radar sensor. The first platform will have a drifting bias applied.



In [ ]:
from stonesoup.calibration.model import Lidar, Lidar2D

In [ ]:
from stonesoup.models.transition.linear import (
    RandomWalk,
    ConstantVelocity,
    OrnsteinUhlenbeck,
    CombinedLinearGaussianTransitionModel,
)
from stonesoup.platform import MovingPlatform, FixedPlatform
from stonesoup.sensor.radar.radar import RadarBearingRange, RadarRotatingBearingRange
from stonesoup.types.state import State, GaussianState
from stonesoup.types.track import Track
from stonesoup.types.array import StateVector
from stonesoup.plotter import Plotterly
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

platforms = [
    FixedPlatform(
        states=State(StateVector([35.0, 15.0]), timestamp=start_time),
        position_mapping=[0, 1],
    ),
    FixedPlatform(
        states=State(StateVector([-20.0, 20.0]), timestamp=start_time),
        position_mapping=[0, 1],
    ),
    FixedPlatform(
        states=State(StateVector([-55.0, 55.0]), timestamp=start_time),
        position_mapping=[0, 1],
    ),
]

sensors = [
    Lidar2D(
        ndim_state=4,
        position_mapping=[0, 2],
        noise_covar=np.diag([0.2, 0.2]) ** 2,
        rpm=40,
        fov_angle=np.radians(270),
        dwell_centre=StateVector([3 * np.pi / 4]),
        max_range=45,
    ),
    Lidar2D(
        ndim_state=4,
        position_mapping=[0, 2],
        noise_covar=np.diag([0.2, 0.2]) ** 2,
        rpm=40,
        fov_angle=np.radians(270),
        dwell_centre=StateVector([np.pi / 2]),
        max_range=45,
    ),
    Lidar2D(
        ndim_state=4,
        position_mapping=[0, 2],
        noise_covar=np.diag([0.2, 0.2]) ** 2,
        rpm=40,
        fov_angle=np.radians(270),
        dwell_centre=StateVector([0.0]),
        max_range=45,
    ),
]

for n, sensor in enumerate(sensors):
    sensor.id = str(n)

Attach Sensors to Platforms



In [ ]:
for platform, sensor in zip(platforms, sensors):
    platform.add_sensor(sensor)

## Add Targets

We add several moving targets to the scenario, each with its own motion model.



In [ ]:
# left to right
targets = {
    MovingPlatform(
        states=State(
            StateVector([-80 + 2 * i, 2, -11 + 7 * i, 0]),
            timestamp=start_time,
        ),
        position_mapping=[0, 2],
        transition_model=CombinedLinearGaussianTransitionModel(
            [ConstantVelocity(0.0005)] * 2
        ),
    )
    # for i in [2, 5]
    for i in [5, 7]
}

# up to down
targets.add(
    MovingPlatform(
        states=State(
            StateVector([-60 + 16, 0, 60, -2]),
            timestamp=start_time,
        ),
        position_mapping=[0, 2],
        transition_model=CombinedLinearGaussianTransitionModel(
            [ConstantVelocity(0.00005)] * 2
        ),
    )
)
targets.add(
    MovingPlatform(
        states=State(
            StateVector([-60 + 72, 0, 110, -2]),
            timestamp=start_time,
        ),
        position_mapping=[0, 2],
        transition_model=CombinedLinearGaussianTransitionModel(
            [ConstantVelocity(0.00005)] * 2
        ),
    )
)


# down to up
# targets.update(
#     {
#         MovingPlatform(
#             states=State(
#                 StateVector([-55 + 8 * i, 0, -50, 2]),
#                 timestamp=start_time,
#             ),
#             position_mapping=[0, 2],
#             transition_model=CombinedLinearGaussianTransitionModel(
#                 [ConstantVelocity(0.00005)] * 2
#             ),
#         )
#         # for i in [2, 5]
#         for i in [2, 9.5]
#     }
# )
from stonesoup.models.transition.linear import KnownTurnRate

# turn_noise_diff_coeffs = np.array([0.0, 0.0])
# turn_rate = np.pi / 20  # specified in radians per seconds...
# turn = KnownTurnRate(turn_noise_diff_coeffs=turn_noise_diff_coeffs, turn_rate=turn_rate)
# # go in a circle


## Simulate Platform Motion and Sensor Measurements

We simulate the motion of each platform and generate sensor measurements for each target.
The first platform's sensor measurements will be affected by a drifting bias.

We create a time-varying bias using a random walk model, and apply this bias to the measurements
of platform 0.



In [ ]:
# x, y, heading (radians) bias
sensor_0_true_bias_prior = State([[-0.4], [0.6], [np.radians(1.0)]], start_time)
sensor_1_true_bias_prior = State([[0.3], [0.2], [np.radians(-1.5)]], start_time)

bias_transition_model = CombinedLinearGaussianTransitionModel(
    [RandomWalk(1e-6), RandomWalk(1e-6), RandomWalk(1e-10)]
)
true_bias_0 = GroundTruthPath([sensor_0_true_bias_prior])
true_bias_1 = GroundTruthPath([sensor_1_true_bias_prior])
bias_truths = [true_bias_0, true_bias_1]

Simulate platforms and measurements including bias for platform 0




In [ ]:
from scipy.spatial.transform import Rotation

ground_truths = [GroundTruthPath() for _ in platforms]

timestamps = [
    start_time + datetime.timedelta(microseconds=n * 200_000) for n in range(1, 500)
]
measurements = [[] for _ in sensors]

for time in timestamps:
    # Update the true bias using the transition model
    for true_bias in bias_truths:
        true_bias.append(
            State.from_state(
                true_bias.state,
                state_vector=bias_transition_model.function(
                    true_bias, noise=True, time_interval=time - true_bias.timestamp
                ),
                timestamp=time,
            )
        )
    for target in targets:
        target.move(timestamp=time)
    for platform_index, platform in enumerate(platforms):
        # sample measurements with accurate noise covariance
        current_measurements = platform.sensors[0].measure(targets, noise=True)
        for measurement in current_measurements:
            # consolidate this infomation. should only be needed once.
            # measurement.metadata["id"] = platform.sensors[0].id
            measurement.metadata["sensorId"] = platform.sensors[0].id

        # Generate measurement for each platform
        measurements[platform_index].append((time, current_measurements))
        # Apply drifting bias to platform 0's sensor measurements
        if platform_index in [0, 1]:
            true_bias = bias_truths[platform_index]
            for model in {m.measurement_model for m in current_measurements}:
                model.translation_offset = (
                    model.translation_offset + true_bias.state_vector[:2]
                )
                biased_rotation = Rotation.from_euler(
                    "xyz", model.rotation_offset.flatten(), degrees=False
                ) * Rotation.from_euler("z", true_bias.state_vector[2], degrees=False)
                model.rotation_offset = biased_rotation.as_euler(
                    "xyz", degrees=False
                ).reshape(-1, 1)
                # inflate noise covariance to account for the bias
                # model.noise_covar = np.diag([2, 2])

## Visualise Ground Truths and Measurements
We plot the ground truth positions of platforms and targets, and the sensor measurements
(with bias for platform 0 in green).



In [ ]:
plotter = Plotterly()
for n, platform in enumerate(platforms):
    plotter.plot_sensors(
        [platform], mapping=[0, 1], line_dash="solid", label=f"sensor {n}"
    )
plotter.plot_ground_truths(targets, mapping=[0, 2])
for n, sensor_measurements in enumerate(measurements):
    kwargs = {}
    if n == 0:
        kwargs["marker"] = {"color": "green"}
    elif n == 1:
        kwargs["marker"] = {"color": "red"}
    elif n == 2:
        kwargs["marker"] = {"color": "blue"}

    plotter.plot_measurements(
        {m for ms in sensor_measurements for m in ms[1]},
        mapping=[0, 2],
        label=f"Sensor {n}",
        **kwargs,
    )

In [ ]:
from stonesoup.sensor.sensor import Sensor
from stonesoup.functions import pol2cart

import plotly.graph_objects as go


def plot_static_sensor_fov(fig, sensors: set[Sensor] | Sensor, default_range=40):
    """
    Plots the current state of a set of sensors onto a static pyplot plot.

    Parameters
    ----------
    fig : :class:`~stonesoup.plotter.Plotter`
        Plotter object to place the fov on.
    sensors : Union[set, Sensor]
        A set of sensors for which to plot the fov's.

    Returns
    -------
    :class:`~stonesoup.plotter.Plotter`
        The completed Plotter object.
    """
    # fig.ax.set_autoscale_on(False)
    if isinstance(sensors, Sensor):
        sensors = {sensors}
    for sensor in sensors:
        x = [0, 0]
        y = [0, 0]
        range_ = getattr(sensor, "max_range")
        for i, fov_side in enumerate((-1, 1)):
            x[i], y[i] = (
                pol2cart(
                    range_, sensor.dwell_centre[0, 0] + sensor.fov_angle / 2 * fov_side
                )
                + sensor.position[[0, 1], 0]
            )

        angles = np.linspace(
            sensor.dwell_centre[0, 0] - sensor.fov_angle / 2,
            sensor.dwell_centre[0, 0] + sensor.fov_angle / 2,
            num=50,
            endpoint=False,
        )

        x = (
            [sensor.position[0], x[0]]
            + [sensor.position[0] + range_ * np.cos(angle) for angle in angles]
            + [x[1], sensor.position[0]]
        )
        y = (
            [sensor.position[1], y[0]]
            + [sensor.position[1] + range_ * np.sin(angle) for angle in angles]
            + [y[1], sensor.position[1]]
        )

        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode="lines",
                name="Sensor FOV",
                line=dict(color="black", dash="dash", width=1),
                showlegend=True,
            )
        )
    return fig

In [ ]:
sensors = set()
for platform in platforms:
    sensors.update(platform.sensors)
plot_static_sensor_fov(plotter.fig, sensors)
plotter.fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)
plotter.fig

In [ ]:
from stonesoup.serialise import YAML
from stonesoup.feeder.multi import MultiDataFeeder
from collections import defaultdict


bias_prior_state = GaussianState(
    [[0.0], [0.0], [0.0]],
    np.diag([0.5**2, 0.5**2, np.radians(1.0) ** 2]),
    timestamp=None,
)
bias_track_store = defaultdict(lambda: Track(copy.copy(bias_prior_state)))
# fixed sensors must have their bias track set to None
bias_track_store["2"] = None
feeder = MultiDataFeeder(measurements[::-1])

tracker_config_path = "bias_tracker_config.yaml"
yaml = YAML()

with open(tracker_config_path) as config:
    config = yaml.load(config)
bias_tracker = config["tracker"]
bias_tracker.calibration_track_selector.fixed_sensor_ids = ["2"]
bias_tracker.detector = feeder
bias_tracker.bias_track_store = bias_track_store

In [ ]:
all_detections = set()
all_tracks = set()
for n, (time, tracks) in enumerate(bias_tracker):
    if n % 100 == 0:
        print(n)
    all_tracks.update(tracks)
    all_detections.update(bias_tracker.detector.detections)

## Visualise Tracking Results

We plot the estimated tracks alongside the ground truths and measurements, showing the effect
of bias estimation.

By comparing the green biased detection to the previous plot (with ground truth layer also to
make comparison clearer), it can be seen that the bias has been corrected.



In [ ]:
from collections import defaultdict
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(timesteps=timestamps)
# for n, platform in enumerate(platforms):
#     plotter.plot_sensors(
#         [platform], mapping=[0, 1], line_dash="solid", label=f"sensor {n}"
#     )
plotter.plot_ground_truths({t.ground_truth_path for t in targets}, mapping=[0, 2])
ordered_dets = defaultdict(set)
for detection in all_detections:
    sensor_id = detection.metadata["sensorId"]
    ordered_dets[sensor_id].add(detection)


for sensor_id, sensor_detections in ordered_dets.items():
    kwargs = {}
    if sensor_id == "0":
        kwargs["marker"] = {"color": "green"}
    elif sensor_id == "1":
        kwargs["marker"] = {"color": "red"}
    elif sensor_id == "2":
        kwargs["marker"] = {"color": "blue"}

    plotter.plot_measurements(
        sensor_detections,
        mapping=[0, 2],
        label="Detections",
        **kwargs,
    )

plot_static_sensor_fov(plotter.fig, sensors)
plotter.fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

single_track = set()
for n, track in enumerate(all_tracks):
    if n == 1:
        single_track.add(track)

single_track = all_tracks
plotter.plot_tracks(
    single_track,
    mapping=[0, 2],
    uncertainty=False,
    label="tracks",
)
plotter.fig

### Current problems:
- possible to calibrate of tracks only being contributed by themselves, or other sensors that have been calibrated from this sensor. Need to establish a hierarchy of sensors that can be calibrated from one another.
- need to have a shared bias track store between many objects. Make a custom defaultdict wrapper that is a singleton?

## Visualise Bias Estimation

Finally, we plot the true bias and the estimated bias over time, for both x and y components,
including 1 standard deviation error area.



In [ ]:
plotter = Plotterly(dimension=1, axis_labels=["Time", "Bias"])
for true_bias in bias_truths:
    plotter.plot_ground_truths(true_bias, mapping=[0], label="True 𝑥 bias")
    plotter.plot_ground_truths(true_bias, mapping=[1], label="True 𝑦 bias")
    plotter.plot_ground_truths(true_bias, mapping=[2], label="True 𝑦aw bias")

for n, bias_track in enumerate(bias_tracker.bias_track_store.values()):
    if bias_track is None:
        continue
    # bias track calculates the offset - so negative bias track is the bias (not true if we're doing 3d rotations... but we're not.)
    for state in bias_track:
        state.state_vector = -state.state_vector

    plotter.plot_tracks(
        bias_track, mapping=[0], label=f"sensor{n}_𝑥 bias estimate", uncertainty=False
    )
    plotter.plot_tracks(
        bias_track, mapping=[1], label=f"sensor{n}_y bias estimate", uncertainty=False
    )
    plotter.plot_tracks(
        bias_track, mapping=[2], label=f"sensor{n}_yaw bias estimate", uncertainty=False
    )


plotter.fig

In [ ]:
bias_track[0]